## Load and preprocess data

In [143]:
import pandas as pd
import re

# Load dataset
df = pd.read_csv("./Dataset/train.csv", encoding = 'ISO-8859-1')
df = df[['text', 'sentiment']].dropna()

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"<.*?>", '', text)
    text = re.sub(r"[^a-z\s]", '', text)
    return re.sub(r'\s+', ' ', text).strip()

df['cleaned_text'] = df['text'].apply(clean_text)


## Encode Labels

In [144]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])  # 0: negative, 1: neutral, 2: positive


## Split Dataset

In [145]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)


## Train & Evaluate with Bag of Words (BoW)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

bow_vectorizer = CountVectorizer(max_features=5000) #The CountVectorizer will consider only the top 5000 most frequent words in the dataset.
# Less frequent words will be ignored (i.e., they won’t appear in the final vocabulary or feature matrix).

X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow = bow_vectorizer.transform(X_test_text)

model_bow = LogisticRegression(max_iter=1000)
model_bow.fit(X_train_bow, y_train)
y_pred_bow = model_bow.predict(X_test_bow)

print("Bag of Words")
print("Accuracy:", accuracy_score(y_test, y_pred_bow))
print(classification_report(y_test, y_pred_bow, target_names=le.classes_))


Bag of Words
Accuracy: 0.6852256186317321
              precision    recall  f1-score   support

    negative       0.70      0.62      0.66      1556
     neutral       0.62      0.71      0.67      2223
    positive       0.77      0.71      0.74      1717

    accuracy                           0.69      5496
   macro avg       0.70      0.68      0.69      5496
weighted avg       0.69      0.69      0.69      5496



## Train & Evaluate with TF-IDF

In [147]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

print("TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print(classification_report(y_test, y_pred_tfidf, target_names=le.classes_))


TF-IDF
Accuracy: 0.6837700145560408
              precision    recall  f1-score   support

    negative       0.71      0.61      0.66      1556
     neutral       0.61      0.74      0.67      2223
    positive       0.79      0.68      0.73      1717

    accuracy                           0.68      5496
   macro avg       0.70      0.68      0.69      5496
weighted avg       0.70      0.68      0.68      5496



## Train & Evaluate with Word Embeddings

### Load GloVe

In [148]:
import numpy as np

# Load GloVe vectors
def load_glove(path):
    embeddings = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vec = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vec
    return embeddings

glove_path = "./glove.6B/glove.6B.100d.txt"
glove = load_glove(glove_path)


### Average Word Vectors

In [149]:
from tqdm import tqdm

def vectorize_text(texts, embeddings, dim=100):
    vectors = []
    for text in tqdm(texts):
        words = text.split()
        vecs = [embeddings[w] for w in words if w in embeddings]
        if vecs:
            vectors.append(np.mean(vecs, axis=0))
        else:
            vectors.append(np.zeros(dim))
    return np.array(vectors)

X_train_embed = vectorize_text(X_train_text, glove)
X_test_embed = vectorize_text(X_test_text, glove)


100%|██████████| 5496/5496 [00:00<00:00, 23284.30it/s]


### Train Logistic Regression

In [150]:
model_embed = LogisticRegression(max_iter=1000)
model_embed.fit(X_train_embed, y_train)
y_pred_embed = model_embed.predict(X_test_embed)

print("Word Embeddings (GloVe Avg)")
print("Accuracy:", accuracy_score(y_test, y_pred_embed))
print(classification_report(y_test, y_pred_embed, target_names=le.classes_))


Word Embeddings (GloVe Avg)
Accuracy: 0.607532751091703
              precision    recall  f1-score   support

    negative       0.64      0.51      0.57      1556
     neutral       0.55      0.69      0.61      2223
    positive       0.69      0.59      0.63      1717

    accuracy                           0.61      5496
   macro avg       0.62      0.60      0.61      5496
weighted avg       0.62      0.61      0.61      5496

